# 1. Lectura e inspección de datos Bronze
Objetivo: consultar la estructura de la tabla que construimos en el Notebook anterior.

Ejecuta:

In [0]:
USE CATALOG github_data;
DESCRIBE TABLE github_data.bronze.github_users_raw

col_name,data_type,comment
login,string,null
id,bigint,null
node_id,string,null
avatar_url,string,null
gravatar_id,string,null
url,string,null
html_url,string,null
followers_url,string,null
following_url,string,null
gists_url,string,null


In [0]:
--Inspección de datos
SELECT *
FROM bronze.github_users_raw
LIMIT 5

login,id,node_id,avatar_url,gravatar_id,url,html_url,followers_url,following_url,gists_url,starred_url,subscriptions_url,organizations_url,repos_url,events_url,received_events_url,type,user_view_type,site_admin,ingestion_timestamp
mojombo,1,MDQ6VXNlcjE=,https://avatars.githubusercontent.com/u/1?v=4,,https://api.github.com/users/mojombo,https://github.com/mojombo,https://api.github.com/users/mojombo/followers,https://api.github.com/users/mojombo/following{/other_user},https://api.github.com/users/mojombo/gists{/gist_id},https://api.github.com/users/mojombo/starred{/owner}{/repo},https://api.github.com/users/mojombo/subscriptions,https://api.github.com/users/mojombo/orgs,https://api.github.com/users/mojombo/repos,https://api.github.com/users/mojombo/events{/privacy},https://api.github.com/users/mojombo/received_events,User,public,false,2026-11-01T21:15:51.444Z
defunkt,2,MDQ6VXNlcjI=,https://avatars.githubusercontent.com/u/2?v=4,,https://api.github.com/users/defunkt,https://github.com/defunkt,https://api.github.com/users/defunkt/followers,https://api.github.com/users/defunkt/following{/other_user},https://api.github.com/users/defunkt/gists{/gist_id},https://api.github.com/users/defunkt/starred{/owner}{/repo},https://api.github.com/users/defunkt/subscriptions,https://api.github.com/users/defunkt/orgs,https://api.github.com/users/defunkt/repos,https://api.github.com/users/defunkt/events{/privacy},https://api.github.com/users/defunkt/received_events,User,public,false,2026-11-01T21:15:51.444Z
pjhyett,3,MDQ6VXNlcjM=,https://avatars.githubusercontent.com/u/3?v=4,,https://api.github.com/users/pjhyett,https://github.com/pjhyett,https://api.github.com/users/pjhyett/followers,https://api.github.com/users/pjhyett/following{/other_user},https://api.github.com/users/pjhyett/gists{/gist_id},https://api.github.com/users/pjhyett/starred{/owner}{/repo},https://api.github.com/users/pjhyett/subscriptions,https://api.github.com/users/pjhyett/orgs,https://api.github.com/users/pjhyett/repos,https://api.github.com/users/pjhyett/events{/privacy},https://api.github.com/users/pjhyett/received_events,User,public,false,2026-11-01T21:15:51.444Z
wycats,4,MDQ6VXNlcjQ=,https://avatars.githubusercontent.com/u/4?v=4,,https://api.github.com/users/wycats,https://github.com/wycats,https://api.github.com/users/wycats/followers,https://api.github.com/users/wycats/following{/other_user},https://api.github.com/users/wycats/gists{/gist_id},https://api.github.com/users/wycats/starred{/owner}{/repo},https://api.github.com/users/wycats/subscriptions,https://api.github.com/users/wycats/orgs,https://api.github.com/users/wycats/repos,https://api.github.com/users/wycats/events{/privacy},https://api.github.com/users/wycats/received_events,User,public,false,2026-11-01T21:15:51.444Z
ezmobius,5,MDQ6VXNlcjU=,https://avatars.githubusercontent.com/u/5?v=4,,https://api.github.com/users/ezmobius,https://github.com/ezmobius,https://api.github.com/users/ezmobius/followers,https://api.github.com/users/ezmobius/following{/other_user},https://api.github.com/users/ezmobius/gists{/gist_id},https://api.github.com/users/ezmobius/starred{/owner}{/repo},https://api.github.com/users/ezmobius/subscriptions,https://api.github.com/users/ezmobius/orgs,https://api.github.com/users/ezmobius/repos,https://api.github.com/users/ezmobius/events{/privacy},https://api.github.com/users/ezmobius/received_events,User,public,false,2026-11-01T21:15:51.444Z


#2.Creación de la tabla silver 

In [0]:
CREATE OR REPLACE TABLE github_data.silver.github_users AS (

    SELECT DISTINCT
        cast(id as bigint) AS user_id,
        TRIM(login)  AS username,
        UPPER(TRIM(Type)) AS account_type ,
        html_url AS profile_url,
        avatar_url,
        COALESCE(site_admin, FALSE) AS is_site_admin,
        ingestion_timestamp


        FROM github_data.bronze.github_users_raw

        WHERE ID IS NOT NULL
            AND login is NOT NULL
            AND TRIM(login) <> ''

);

SELECT *
FROM github_data.silver.github_users
LIMIT 2;

user_id,username,account_type,profile_url,avatar_url,is_site_admin,ingestion_timestamp
3,pjhyett,USER,https://github.com/pjhyett,https://avatars.githubusercontent.com/u/3?v=4,false,2026-11-01T21:15:51.444Z
18,wayneeseguin,USER,https://github.com/wayneeseguin,https://avatars.githubusercontent.com/u/18?v=4,false,2026-11-01T21:15:51.444Z


# 3. Validaciones de calidad – Data Quality

OBJETIVO

Comprobar que los datos almacenados en Silver cumplen las reglas mínimas de calidad que definimos 

In [0]:
SELECT
    user_id, 
    count(*) as occurrences
FROM github_data.silver.github_users
GROUP BY user_id
HAVING count(*)  > 1
ORDER BY user_id ASC



user_id,occurrences


In [0]:
--validación de valores nulos

select

    SUM(CASE WHEN user_id is null then 1 else  0 END) as Id_nulos,
    SUM(CASE WHEN username is NULL THEN 1 ELSE 0 END)  as Username_Nulos,
    SUM(CASE WHEN profile_url IS NULL THEN 1 ELSE 0 END) AS null_profile_url

FROM github_data.silver.github_users

Id_nulos,Username_Nulos,null_profile_url
0,0,0


In [0]:
--Validar consistencia de tipos de cuenta
SELECT
    account_type,
    COUNT(*) AS total_users
FROM github_data.silver.github_users
GROUP BY account_type
ORDER BY total_users DESC;

account_type,total_users
USER,10
